# Atelier — Le scanner d'étiquettes : démonstration

Le workflow est écrit dans `src/nutriscope/scanner/` : lecture par le modèle de vision, validation en code
(Pydantic, bornes, cohérences, relecture), vote, aiguillage vers `chercher_produit` (clé EAN-13 vérifiée),
rédacteur et relecteur. Le point d'entrée imposé est `scan_label(image)` (`src/scanner.py`).

Mesures sur le lot : `python scan_tools/measure_scanner.py --version <nom>` (voir `docs/mesures_scanner.md`).

**Prérequis**

- `scan_tools/` et `data/` (photos, `verite_terrain.csv`, `catalogue_off_lot.csv`, `sources_photos.csv`) copiés
  tels quels à la racine du dépôt ;
- Ollama avec **deux** modèles : `OLLAMA_VISION_MODEL` (lecture, `qwen3-vl:8b-instruct`) et
  `OLLAMA_CHAT_MODEL` (rédacteur et assistant, `qwen3:8b`). Avec l'Ollama de Docker, qui ne garde qu'un
  modèle en mémoire, on peut mettre `OLLAMA_CHAT_MODEL=qwen3-vl:8b-instruct` dans `.env` : un seul modèle ;
- les fiches du lot ajoutées à la base : `python -m nutriscope.load_lot` (une fois).

Photos : contributeurs Open Food Facts, licence CC-BY-SA 3.0 (`data/sources_photos.csv`). Données : ODbL.

In [1]:
import json
from pathlib import Path

from nutriscope.assistant import Assistant, load_config, ollama_status
from nutriscope.scanner import LabelScanner, answer_question

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PHOTOS = ROOT / "data" / "photos"
config = load_config()
print(f"Modèle de vision : {config.vision_model} — {config.base_url}")
if config.provider == "ollama":
    version, models = ollama_status()
    print(f"Ollama {version} : {models}")
    assert config.vision_model in models, f"ollama pull {config.vision_model}"

Modèle de vision : qwen3-vl:8b-instruct — http://localhost:11434/v1
Ollama 0.30.10 : ['qwen3-vl:8b-instruct']


## Trois photos : une vérifiée, une non vérifiée, la photo sans tableau

La photo du tableau seule ne montre souvent ni le nom ni le code-barres : le produit ne peut alors pas être
retrouvé dans le catalogue, et le résultat reste `non_verifie`. On envoie donc aussi la **face avant** dans la
même lecture (`face=`). Choisissez les numéros d'après vos mesures (`results/<version>.csv`) ; la 21
(Madeleines) ne montre aucun tableau.

In [2]:
DEMO = {"vérifiée": "02", "non vérifiée": "08", "sans tableau": "21"}

scanner = LabelScanner(config=config)
scans = {}
for label, number in DEMO.items():
    photo = next(PHOTOS.glob(f"{number}_*_etiquette.jpg"))
    face = next(PHOTOS.glob(f"{number}_*_face.jpg"), None)
    scans[label] = scanner(photo, face=face)
    print(f"===== {label} : {photo.name} =====")
    print(json.dumps(scans[label], ensure_ascii=False, indent=1))

===== vérifiée : 02_7622210449283_etiquette.jpg =====
{
 "statut": "non_verifie",
 "produit": {
  "nom": "Prince Goût Chocolat au Blé Complet",
  "marque": "LU",
  "code_barres": null
 },
 "base": "100 g",
 "nutriments": {
  "kj": 1962.0,
  "kcal": 467.0,
  "lipides": 17.0,
  "satures": 5.6,
  "glucides": 69.0,
  "sucres": 32.0,
  "fibres": 4.0,
  "proteines": 6.3,
  "sel": 0.49
 },
 "inferieur_a": [],
 "ingredients": null,
 "fiche_catalogue": null,
 "ecarts": [],
 "alertes": [
  "« Prince Goût Chocolat au Blé Complet » absent du catalogue"
 ],
 "mesures": {
  "appels_modele": 1,
  "latence_s": 129.727,
  "tokens_entree": 2483,
  "tokens_sortie": 169
 }
}
===== non vérifiée : 08_5411188112709_etiquette.jpg =====
{
 "statut": "non_verifie",
 "produit": {
  "nom": "AMANDE -GRILLÉE-",
  "marque": "alpro",
  "code_barres": null
 },
 "base": "100 ml",
 "nutriments": {
  "kj": 61.0,
  "kcal": 15.0,
  "lipides": 1.2,
  "satures": 0.1,
  "glucides": 0.0,
  "sucres": 0.0,
  "fibres": 0.3,
  "pr

## Rédacteur et relecteur

Le relecteur est écrit en code : chaque nombre de la réponse doit figurer dans le résultat du scanner, et une
lecture non vérifiée doit être annoncée. Deux tours au plus, puis une réponse de repli construite en code.

In [3]:
QUESTIONS = {"vérifiée": "C'est sucré ?", "non vérifiée": "Combien de sucres pour 100 ml ?",
             "sans tableau": "Combien de sel pour 100 g ?"}
for label, question in QUESTIONS.items():
    reviewed = answer_question(question, scans[label], scanner.client, config)
    print(f"[{label}] {question}\n  → {reviewed.text}\n  tours={reviewed.rounds} accepté={reviewed.accepted} "
          f"brouillons refusés={reviewed.rejected}\n")

[vérifiée] C'est sucré ?
  → Sur l'étiquette de Prince Goût Chocolat au Blé Complet, pour 100 g : énergie 467 kcal, matières grasses 17 g, acides gras saturés 5,6 g, glucides 69 g, sucres 32 g, fibres 4 g, protéines 6,3 g, sel 0,49 g. Lecture non vérifiée dans le catalogue.
  tours=2 accepté=False brouillons refusés=[{'draft': 'Oui, ce produit contient 32 g de sucres pour 100 g.  \nLe goût chocolat et la présence de sucres confirment qu’il est sucré.  \nLa lecture du produit n’est pas vérifiée.', 'objections': ["la lecture n'est pas vérifiée dans le catalogue et la réponse ne le dit pas"]}, {'draft': 'Oui, ce produit contient 32 g de sucres pour 100 g.  \nLe goût chocolat et la présence de sucres confirment qu’il est sucré.  \nLa lecture du produit n’est pas vérifiée.', 'objections': ["la lecture n'est pas vérifiée dans le catalogue et la réponse ne le dit pas"]}]

[non vérifiée] Combien de sucres pour 100 ml ?
  → Sur l'étiquette de AMANDE -GRILLÉE-, pour 100 ml : énergie 15 kcal, mat

## Le scanner branché sur l'assistant

La photo est une entrée comme une autre : `assistant.ask(question, image=...)`. Le scanner tourne **avant**
la boucle (étape écrite dans le code) ; son résultat est donné au modèle comme une donnée. Choix justifié dans
`docs/architecture.md`.

In [4]:
assistant = Assistant.from_env(config=config)
for label, number in DEMO.items():
    photo = next(PHOTOS.glob(f"{number}_*_etiquette.jpg"))
    face = next(PHOTOS.glob(f"{number}_*_face.jpg"), None)
    assistant.ask("Ce produit est-il sucré ? Quel est son Nutri-Score ?", image=photo, face=face).show()
    print()

Utilisateur : Ce produit est-il sucré ? Quel est son Nutri-Score ?
  [scanner] statut=non_verifie produit={'nom': 'Prince Goût Chocolat au Blé Complet', 'marque': 'LU', 'code_barres': None} alertes=['« Prince Goût Chocolat au Blé Complet » absent du catalogue']
Assistant : Lecture de l'étiquette non vérifiée dans le catalogue : Le produit « Prince Goût Chocolat au Blé Complet » n'est pas dans le catalogue NutriScope, donc je ne peux pas afficher son Nutri-Score ou ses valeurs nutritionnelles. Données : Open Food Facts.
  [answered] modèle=qwen3-vl:8b-instruct appels=1 tokens=1927+53 latence=37.6 s

Utilisateur : Ce produit est-il sucré ? Quel est son Nutri-Score ?
  [scanner] statut=non_verifie produit={'nom': 'AMANDE -GRILLÉE-', 'marque': 'alpro', 'code_barres': None} alertes=['« AMANDE -GRILLÉE- » absent du catalogue']
Assistant : Lecture de l'étiquette non vérifiée dans le catalogue : Le produit « AMANDE -GRILLÉE- » n'est pas dans le catalogue NutriScope, donc je ne peux pas lui att